# Create VQA dataset from CLEVR

In [ ]:
import json
import string
import random
from os.path import join

## behaviour

In [ ]:
behaviour = 'count'
behaviour = 'colour'

In [ ]:
def gen_true_answers() -> list:
    if behaviour == 'count':
        return [str(i) for i in range(11)]
    if behaviour == 'colour':
        return ['gray', 'red', 'blue', 'green', 'brown', 'purple', 'cyan', 'yellow']
    return ['2']

In [ ]:
beh_answers = gen_true_answers()
beh_answers

In [ ]:
uppercase_alphabet = string.ascii_uppercase
def gen_answ_options() -> str:
    suffix = '\nChoices:'
    for i, answ in enumerate(beh_answers):
        suffix += f'\n{uppercase_alphabet[i]} {answ}'
    return suffix

In [ ]:
anw_options = gen_answ_options()
anw_options

In [ ]:
def map_answ_option(a: str) -> str:
    return uppercase_alphabet[beh_answers.index(a)]

In [ ]:
def behaviour_cond(true_answer) -> bool:
    return str(true_answer) in beh_answers

In [ ]:
def gen_target_answ(true_answer: str) -> str:
    if behaviour == 'count':
        return str(random.choice([beh_answers.copy().remove(true_answer)]))
    if behaviour == 'colour':
        return str(random.choice([beh_answers.copy().remove(true_answer)]))
    return '4'

## src data

In [ ]:
clvr_pth = '/informatik/wtm/datasets/External Datasets/CLEVR/CLEVR_v1.0'
data_dir = '/data/5drwal'
data_dir = '/export/home/5drwal/infhome/Documents/NN/caa'

## code

In [ ]:
def prepare_counting_clevr_dss(clevr_qs_pth, out_pth, nos_qs=[1000], val=False):
    with open(clevr_qs_pth, 'r') as f:
        clevr_data = json.load(f)
    
    questions = clevr_data['questions']
    contrastive_pairs = [] # no answ options provided in q
    contrastive_pairs_options = [] # answ options provided in q
    random.shuffle(questions)
    
    for q in questions:
        true_answer = str(q['answer'])
        b_cond = behaviour_cond(true_answer)
        if val or b_cond:
            contrastive_pairs.append({
                "q_type": behaviour if b_cond else 'other',
                "image_path": q['image_filename'],
                "question": q['question'],
                "true_answer": true_answer,
                "target_answer": gen_target_answ(true_answer) if b_cond else true_answer
            })
            contrastive_pairs_options.append({
                "q_type": behaviour if b_cond else 'other',
                "image_path": q['image_filename'],
                "question": q['question'] + anw_options,
                "true_answer": map_answ_option(true_answer),
                "target_answer": map_answ_option(gen_target_answ(true_answer)) if b_cond else map_answ_option(true_answer)
            })

    for no_qs in nos_qs:
        with open(f'{out_pth}_{no_qs}.jsonl', 'w') as f:
            for pair in contrastive_pairs[:min(no_qs, len(contrastive_pairs))]:
                f.write(json.dumps(pair) + '\n')
            print('done', len(contrastive_pairs), out_pth, no_qs)
            
    for no_qs in nos_qs:
        with open(f'{out_pth}_opt_{no_qs}.jsonl', 'w') as f: # opt - answ options provided in q
            for pair in contrastive_pairs_options[:min(no_qs, len(contrastive_pairs_options))]:
                f.write(json.dumps(pair) + '\n')
            print('done', len(contrastive_pairs_options), out_pth, no_qs)

In [ ]:
def save_subsets(clevr_qs_pth, out_pth, splt='train', nos_qs=[1000], val=False):
    prepare_counting_clevr_dss(clevr_qs_pth, f'{out_pth}_{splt}', nos_qs, val)

## save

In [ ]:
in_pth = join(clvr_pth, 'questions/CLEVR_train_questions.json')
out_pth = join(data_dir, f'caa_{behaviour}')
save_subsets(in_pth, out_pth, 'train', [10, 50, 100, 1000])

In [ ]:
in_pth = join(clvr_pth, 'questions/CLEVR_val_questions.json')
out_pth = join(data_dir, f'caa_{behaviour}')
save_subsets(in_pth, out_pth, 'val', [10, 50, 100, 1000], False)

In [ ]:
in_pth = join(clvr_pth, 'questions/CLEVR_val_questions.json')
out_pth = join(data_dir, f'caa_{behaviour}_all')
save_subsets(in_pth, out_pth, 'val', [10, 50, 100, 1000], True)